In [1]:
import torch
import torch.nn as nn
import math

In [2]:
# Example sentence (decoder-only context)
sentence = ["ChatGPT", "is", "amazing", "."]
seq_len = len(sentence)
vocab_size = 10000
embedding_dim = 16
num_heads = 2
hidden_dim = 32

In [3]:
# 1️⃣ Token embeddings
token_embedding = nn.Embedding(vocab_size, embedding_dim)
token_ids = torch.arange(seq_len)
tokens_emb = token_embedding(token_ids)  # [seq_len, embedding_dim]

In [4]:
# 2️⃣ Positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

pos_encoder = PositionalEncoding(embedding_dim)
tokens_emb = pos_encoder(tokens_emb)  # [seq_len, embedding_dim]

In [5]:
# 3️⃣ Decoder-only Transformer layer (masked self-attention)
decoder_layer = nn.TransformerDecoderLayer(
    d_model=embedding_dim,
    nhead=num_heads,
    dim_feedforward=hidden_dim
)
# Wrap in nn.TransformerDecoder with dummy memory (not used)
transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=1)

# Add batch dimension: [seq_len, batch_size, embedding_dim]
tokens_emb = tokens_emb.unsqueeze(1)

# Create causal mask to prevent attending to future tokens
tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len)

In [6]:
# Forward pass (memory=None because decoder-only)
# We pass an empty memory tensor to satisfy nn.TransformerDecoder API
memory = torch.zeros(0, 1, embedding_dim)  # [0, batch_size, d_model]
output = transformer_decoder(tgt=tokens_emb, memory=memory, tgt_mask=tgt_mask)

print("Decoder-only output shape:", output.shape)
print("Decoder-only output:\n", output.squeeze(1))

Decoder-only output shape: torch.Size([4, 1, 16])
Decoder-only output:
 tensor([[-1.1160e-01, -3.2635e-03, -9.9053e-01,  5.7423e-01, -6.0590e-01,
         -4.2570e-01, -2.5864e+00, -2.4386e-02, -1.1669e+00, -1.4866e-01,
          1.1186e+00,  4.2664e-01,  2.6909e-01,  1.0551e+00,  1.2640e+00,
          1.3557e+00],
        [-5.2414e-01, -2.2635e-01,  5.8059e-02,  1.7981e+00, -3.4101e-01,
          1.0435e+00, -3.8480e-01, -2.2012e+00, -1.5042e+00,  5.1934e-02,
          2.1894e-03,  8.1236e-01,  9.8731e-01, -6.5243e-01, -2.7484e-01,
          1.3556e+00],
        [ 8.7243e-02, -1.3347e+00, -3.0294e-01, -9.5937e-01,  3.3964e-01,
         -7.0290e-01,  1.8390e+00,  6.3417e-01, -3.9541e-01,  1.0128e+00,
          7.1739e-01,  8.1413e-01, -2.0124e+00,  1.1987e+00,  4.2831e-02,
         -9.7827e-01],
        [-4.6383e-01, -5.9176e-01, -8.2950e-01,  8.0587e-01,  4.4764e-02,
         -1.8982e-01, -1.7816e+00,  1.3984e-01,  3.2559e-01,  9.7194e-01,
         -2.9913e-01,  1.1961e+00, -2.2441e+0